# FD-24: Дозапуск упавших слотов (Слоты 6, 8, 9) с подключением предыдущего Output
> **Время прогона:** ~1.1–1.2 часа GPU (Слот 6 ~45 мин + Слот 8 ~5 сек + Слот 9 ~25 мин).
> **Inputs блокнота:**
> 1. `freshdoc-dataset` (`dataset_v1.parquet`)
> 2. `Notebook Output` предыдущего запуска (содержит `runs.csv`, `m1.csv`, `slot4_full/best`, `preds.csv`)


In [ ]:
%cd /kaggle/working
import os
from kaggle_secrets import UserSecretsClient
try:
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    _repo = f"https://{_tok}@github.com/o1uh/FreshDoc.git"
    print("clone: private (token)")
except Exception:
    _repo = "https://github.com/o1uh/FreshDoc.git"
    print("clone: public fallback (repo must be public)")

!if [ ! -d FreshDoc ]; then git clone --quiet {_repo} > /tmp/clone.log 2>&1 || (echo "CLONE FAILED:"; sed -E 's#(https?://)[^@]+@#\1***@#g' /tmp/clone.log); fi
if '_tok' in locals(): del _tok
if '_repo' in locals(): del _repo

%cd /kaggle/working/FreshDoc
!git fetch origin feat/FD-24-kaggle-experiments
!git checkout feat/FD-24-kaggle-experiments 2>/dev/null || !git checkout main 2>/dev/null || true
!git pull --ff-only || echo "PULL SKIPPED: убедись, что код свежий"

os.environ["NCCL_P2P_DISABLE"] = "1"
!python -c "import transformers, sys; from packaging import version; sys.exit(0 if version.parse(transformers.__version__) >= version.parse('4.48.0') else 1)" || pip install --quiet --no-deps "transformers>=4.48.0"
!nvidia-smi


In [ ]:
import os, glob, pandas as pd
_cands = glob.glob("/kaggle/input/**/dataset_v1.parquet", recursive=True)
assert _cands, "dataset_v1.parquet не найден в /kaggle/input (добавь датасет в Inputs блокнота)"
_p = _cands[0]
os.environ["DATA_PATH"] = _p
print(f"Dataset path: {_p}")

_df = pd.read_parquet(_p, columns=["split"])
assert len(_df) == 27203, f"dataset rows {len(_df)} != 27203"
print("dataset OK: 27203 rows")
del _df


In [ ]:
# Импорт истории runs.csv и бейзлайнов из предыдущего запуска (inputs)
import os, glob, shutil

# 1. Восстановление runs.csv
local_runs = os.path.abspath('reports/runs.csv')
input_runs = sorted(glob.glob('/kaggle/input/**/runs.csv', recursive=True), key=os.path.getmtime, reverse=True)
if input_runs:
    print(f'Импорт runs.csv из предыдущего запуска: {input_runs[0]} -> {local_runs}')
    shutil.copy(input_runs[0], local_runs)
else:
    print('ПРЕДУПРЕЖДЕНИЕ: runs.csv из предыдущего запуска не найден в inputs!')

# 2. Бейзлайны M0 и M1: копирование или быстрая генерация
input_m1 = glob.glob('/kaggle/input/**/m1.csv', recursive=True)
if input_m1 and os.path.exists(input_m1[0]):
    shutil.copy(input_m1[0], '/kaggle/working/m1.csv')
    input_m0 = glob.glob('/kaggle/input/**/m0.csv', recursive=True)
    if input_m0: shutil.copy(input_m0[0], '/kaggle/working/m0.csv')
    print('Бейзлайны m0.csv и m1.csv скопированы из inputs.')
else:
    print('Генерация m0.csv и m1.csv (~30 сек)...')
    !python scripts/baselines_m0_m1.py --dataset "$DATA_PATH" \
      --out_m0 /kaggle/working/m0.csv --out_m1 /kaggle/working/m1.csv \
      --out_model /kaggle/working/m1_hgb.joblib --out_report /kaggle/working/m1_report.json


## Слот 6: GTE-ModernBERT-base @2048 (снятие усечения + relevance претрейн)
> Исправлено: загрузка в FP32 (`torch_dtype=torch.float32`) для корректной работы с GradScaler FP16.

In [ ]:
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name Alibaba-NLP/gte-modernbert-base \
  --max_len 2048 --gradient_checkpointing --train_mix full \
  --output_dir /kaggle/working/checkpoints/slot6_gte2048 --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов
!rm -rf /kaggle/working/checkpoints/slot6_gte2048/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot6_gte2048/best

# 3. Инференс на test_real и gold
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot6_gte2048/best \
  --dataset "$DATA_PATH" --batch_size 16 --out /kaggle/working/checkpoints/slot6_gte2048/preds.csv

# 4. Оценка
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot6_gte2048/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot6_gte2048/metrics.json --out_table /kaggle/working/checkpoints/slot6_gte2048/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot6_gte2048/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot6_gte2048/table_e1.csv


## Слот 8: Single-split стэкинг M1 + M2 на отложенных предсказаниях `val_real`
> Выполняется на CPU за ~3-5 секунд по готовым предсказаниям лучшего бэкбона.

In [ ]:
import os, glob, subprocess, pandas as pd
df_runs = pd.read_csv('reports/runs.csv')
is_neural = ~df_runs['model_name'].astype(str).str.startswith('stacking')
is_not_ablation = ~df_runs.get('drop_diff', pd.Series(False, index=df_runs.index)).astype(str).str.lower().isin(['true', '1'])
neural_runs = df_runs.loc[is_neural & is_not_ablation].dropna(subset=['eval_pr_auc'])

def _has_ckpt(d):
    name = os.path.basename(str(d))
    return os.path.exists(os.path.join(str(d), 'best')) or os.path.exists(os.path.join(str(d), 'preds.csv')) or \
           bool(glob.glob(f'/kaggle/input/**/{name}/best', recursive=True)) or \
           bool(glob.glob(f'/kaggle/input/**/{name}/preds.csv', recursive=True))

neural_runs = neural_runs.loc[neural_runs['output_dir'].apply(_has_ckpt)]
best_run = neural_runs.sort_values(by=['eval_pr_auc', 'timestamp'], ascending=[False, False]).iloc[0]

best_ckpt = os.path.join(best_run['output_dir'], 'best')
best_preds = os.path.join(best_run['output_dir'], 'preds.csv')
ckpt_name = os.path.basename(best_run['output_dir'])
if not os.path.exists(best_ckpt):
    _cands = glob.glob(f'/kaggle/input/**/{ckpt_name}/best', recursive=True)
    if _cands: best_ckpt = _cands[0]
if not os.path.exists(best_preds):
    _cands_p = glob.glob(f'/kaggle/input/**/{ckpt_name}/preds.csv', recursive=True)
    if _cands_p: best_preds = _cands_p[0]

print(f"Лучший бэкбон для стэкинга: {best_run['model_name']} (PR-AUC: {best_run['eval_pr_auc']:.5f})")
print(f"Источник предсказаний M2: {best_preds}")

data_path = os.environ.get('DATA_PATH')
if os.path.exists(best_preds):
    cmd_stack = (
        f'python scripts/train_stacking.py --dataset "{data_path}" '
        f'--m1_preds /kaggle/working/m1.csv --m2_preds "{best_preds}" '
        f'--output_dir /kaggle/working/checkpoints/slot8_stacking '
        f'--out_preds /kaggle/working/checkpoints/slot8_stacking/m_stack.csv '
        f'--runs_csv reports/runs.csv'
    )
else:
    cmd_stack = (
        f'python scripts/train_stacking.py --dataset "{data_path}" '
        f'--m1_preds /kaggle/working/m1.csv --m2_checkpoint "{best_ckpt}" --batch_size 16 '
        f'--output_dir /kaggle/working/checkpoints/slot8_stacking '
        f'--out_preds /kaggle/working/checkpoints/slot8_stacking/m_stack.csv '
        f'--runs_csv reports/runs.csv'
    )
subprocess.run(cmd_stack, shell=True, check=True)

# Оценка стэкинга через evaluate.py
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv \
  --pred M2=/kaggle/working/checkpoints/slot8_stacking/m_stack.csv \
  --out_metrics /kaggle/working/checkpoints/slot8_stacking/metrics.json \
  --out_table /kaggle/working/checkpoints/slot8_stacking/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot8_stacking/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot8_stacking/table_e1.csv


## Слот 9: E3 — Абляция БЕЗ `diff_code` (`--drop_diff`) на лучшем бэкбоне

In [ ]:
best_model_name = best_run["model_name"]
best_max_len = int(best_run["max_len"])
gc_flag = "--gradient_checkpointing" if best_max_len >= 2048 else ""

data_path = os.environ.get('DATA_PATH')
cmd_train = (
    f'torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml '
    f'--data_path "{data_path}" --model_name "{best_model_name}" '
    f'--max_len {best_max_len} {gc_flag} --drop_diff --train_mix full '
    f'--output_dir /kaggle/working/checkpoints/slot9_ablation_no_diff --runs_csv reports/runs.csv'
)
import subprocess
print(f"Выполнение: {cmd_train}")
subprocess.run(cmd_train, shell=True, check=True)
!rm -rf /kaggle/working/checkpoints/slot9_ablation_no_diff/checkpoint-*

# Инференс с флагом --drop_diff
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot9_ablation_no_diff/best \
  --dataset "$DATA_PATH" --batch_size 16 --drop_diff --out /kaggle/working/checkpoints/slot9_ablation_no_diff/preds.csv

# Оценка абляции
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv \
  --pred M2=/kaggle/working/checkpoints/slot9_ablation_no_diff/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot9_ablation_no_diff/metrics.json \
  --out_table /kaggle/working/checkpoints/slot9_ablation_no_diff/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot9_ablation_no_diff/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot9_ablation_no_diff/table_e1.csv


## Финализация: автоматический выбор лучшего бэкбона по `val_real` PR-AUC

In [ ]:
!python scripts/finalize_best_config.py --runs_csv reports/runs.csv \
  --out_config config/train_best.yaml
!cat config/train_best.yaml

import pandas as pd
df_runs = pd.read_csv("reports/runs.csv")
print("\n=== ИТОГОВАЯ ТАБЛИЦА ВСЕХ ПРОГОНОВ FD-24 ===")
print(df_runs.to_string(index=False))
